# Spine XR Augmentation — Project 3 Colab Runner (ROI-patch pipeline)

Google Colab Pro+ (A100). **Köklü değişiklik:** lezyonlar tüm görüntünün %0.15–3.1'i kadar (74×82 – 350×200 px, ~3000×2500 px görüntülerde). Tüm görüntüyü 224'e küçültünce lezyon ~5px'e iniyor → görünmez. Bu yüzden artık **bbox ROI crop** üzerinde çalışıyoruz: baseline, traditional, WGAN ve hybrid hepsi lezyon yamaları üzerinden.

## Sıralama
1. Bootstrap → 2. Audit+Splits → **2b. ROI patch üret** → 3. ROI Baseline → 4. ROI Traditional → 5. ROI WGAN → 6. Üretim (07) → 7. ROI Hybrid → karar.

**Dürüstlük notu (teze yazılacak):** ROI sınıflandırması 'lokalizasyon verili' varsayar (test'te bbox konumu kullanılır). Bu meşru ama tüm-omurga taramasından farklı bir görevdir; NF yamaları lezyon yama boyut dağılımına eşlenerek 'crop-tightness' kısayolu engellenir (R3).

## 1. Bootstrap

In [ ]:
# 1. Drive Mount
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

# 2. Çalışma Alanını Yerel SSD'de Ayarla (A100'ün maksimum hızı için)
LOCAL_ROOT = Path('/content/spine-xr-augmentation-study')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(LOCAL_ROOT)

# 3. Kodları Drive'dan Yerele Kopyala
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/spine-xr-augmentation-study/spine-xr-augmentation-study')
!cp -r {DRIVE_REPO_PATH}/* .

# 4. Dataset'i SSD'ye Çek ve Aç (Dataset Drive'da .rar olarak durmalı)
# Klasör Adı "dataset" olmalı — configs/base.yaml relatif `dataset/...` yolları kullanır.
DRIVE_DATASET_PATH = Path('/content/drive/MyDrive/spine-xr-augmentation-study/dataset.rar')
!unrar x -o+ {DRIVE_DATASET_PATH} {LOCAL_ROOT}/

# 5. Çıktıların (Outputs) Kaybolmaması İçin Drive'a Bağla
DRIVE_OUTPUTS = Path('/content/drive/MyDrive/spine-xr-augmentation-study/outputs')
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

if os.path.exists('outputs') and not os.path.islink('outputs'):
    shutil.rmtree('outputs')
elif os.path.islink('outputs'):
    os.remove('outputs')
os.symlink(DRIVE_OUTPUTS, 'outputs')

print(f"Çalışma dizini (SSD): {os.getcwd()}")
!ls -l

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 2. Audit + Splits (whole-image image-level split membership)

In [ ]:
!python scripts/01_audit.py --config configs/base.yaml
!python scripts/02_data_splitter.py --config configs/base.yaml --cases configs/cases.yaml
!cat outputs/02_splits/splits_summary.md

## 2b. ROI patch üretimi (bbox crop)

Her case ve split için: abnormal görüntülerin bbox'larından lezyon yamaları + NF görüntülerden eşit-boyutlu rastgele yamalar. 02_splits'teki split üyeliğini kullanır (yeni sızıntı yok).

In [ ]:
!python scripts/02b_build_roi_patches.py --config configs/base.yaml --cases configs/cases.yaml
!cat outputs/02b_roi/summary.md

### 2b.1 QA — pozitif vs NF yamalarını gözle kontrol et (boyutlar eşleşmeli, lezyon ortada)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, cv2
df = pd.read_csv('outputs/02b_roi/case_1/train.csv')
pos = df[df.source=='abnormal_roi'].sample(4, random_state=0)
nf  = df[df.source=='nf_roi'].sample(4, random_state=0)
fig,ax=plt.subplots(2,4,figsize=(12,6))
for j,(_,r) in enumerate(pos.iterrows()):
    ax[0,j].imshow(cv2.imread(r.path,0),cmap='gray'); ax[0,j].set_title(f"POS {r.lesion_type[:12]}"); ax[0,j].axis('off')
for j,(_,r) in enumerate(nf.iterrows()):
    ax[1,j].imshow(cv2.imread(r.path,0),cmap='gray'); ax[1,j].set_title('NF'); ax[1,j].axis('off')
plt.tight_layout(); plt.show()

## 3. ROI Baseline (no aug) — train/val/test hepsi ROI yamaları

`--splits-tag 02b_roi` → train, internal_val, test hepsi 02b_roi'den okunur. Out-tag `03_roi_baseline` (eski whole-image `03_baseline` korunur, tezde kıyas için).

### 3.0 Smoke (case_4 / vgg16 / 1 epoch)

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --splits-tag 02b_roi --cases-filter case_4 --backbones-filter vgg16 \
    --epochs 1 --out-tag 03_roi_smoke

### 3.1 case_1 (VGG16 + InceptionV3)

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --splits-tag 02b_roi --cases-filter case_1 \
    --out-tag 03_roi_baseline

### 3.2 case_2 (VGG16 + InceptionV3)

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --splits-tag 02b_roi --cases-filter case_2 \
    --out-tag 03_roi_baseline

### 3.3 case_3 (VGG16 + InceptionV3)

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --splits-tag 02b_roi --cases-filter case_3 \
    --out-tag 03_roi_baseline

### 3.4 case_4 (VGG16 + InceptionV3)

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --splits-tag 02b_roi --cases-filter case_4 \
    --out-tag 03_roi_baseline

## 4. ROI Traditional augmentation

Lezyon yamalarına 3 transform (rot270 + shear30 + case 2.rot). NF yamaları artırılmaz.

### 4.1 Build

In [ ]:
!python scripts/04_build_traditional_set.py \
    --config configs/base.yaml --cases configs/cases.yaml \
    --splits-tag 02b_roi --out-tag 04_roi_traditional
!cat outputs/04_roi_traditional/summary.md

### 4.1 case_1 (VGG16 + InceptionV3)

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_roi_traditional --train-csv-name train_traditional.csv \
    --splits-tag 02b_roi --cases-filter case_1 \
    --out-tag 04_roi_traditional

### 4.2 case_2 (VGG16 + InceptionV3)

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_roi_traditional --train-csv-name train_traditional.csv \
    --splits-tag 02b_roi --cases-filter case_2 \
    --out-tag 04_roi_traditional

### 4.3 case_3 (VGG16 + InceptionV3)

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_roi_traditional --train-csv-name train_traditional.csv \
    --splits-tag 02b_roi --cases-filter case_3 \
    --out-tag 04_roi_traditional

### 4.4 case_4 (VGG16 + InceptionV3)

In [ ]:
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --train-csv-root outputs/04_roi_traditional --train-csv-name train_traditional.csv \
    --splits-tag 02b_roi --cases-filter case_4 \
    --out-tag 04_roi_traditional

## 5. ROI WGAN (paper-faithful weight clipping, 128² lezyon yamaları)

Her minority sınıf için WGAN, lezyon yamalarında eğitilir. Latent 120, RMSProp 5e-5, n_critic=5, clip 0.01, 12000 iter, snapshot 3000'den itibaren her 1000'de. Osteophytes hariç (majority).

### 5.0 Smoke (Vertebral collapse, 600 iter) — loss O(1) olmalı

In [ ]:
!python scripts/05_train_wgan.py --config configs/base.yaml --wgan configs/wgan.yaml \
    --classes-filter "Vertebral collapse" --iterations 600 --out-tag 05_roi_smoke
from pathlib import Path
import matplotlib.pyplot as plt, cv2
s=sorted(Path('outputs/05_roi_smoke/case_1/vertebral_collapse/samples').glob('*.png'))
if s: plt.figure(figsize=(6,6)); plt.imshow(cv2.imread(str(s[-1]),0),cmap='gray'); plt.axis('off'); plt.show()

### 5.1 WGAN — Disc space narrowing (case_1)

In [ ]:
!python scripts/05_train_wgan.py --config configs/base.yaml --wgan configs/wgan.yaml \
    --classes-filter "Disc space narrowing" --out-tag 05_roi_wgan

### 5.2 WGAN — Vertebral collapse (case_1)

In [ ]:
!python scripts/05_train_wgan.py --config configs/base.yaml --wgan configs/wgan.yaml \
    --classes-filter "Vertebral collapse" --out-tag 05_roi_wgan

### 5.3 WGAN — Foraminal stenosis (case_2)

In [ ]:
!python scripts/05_train_wgan.py --config configs/base.yaml --wgan configs/wgan.yaml \
    --classes-filter "Foraminal stenosis" --out-tag 05_roi_wgan

### 5.4 WGAN — Spondylolysthesis (case_2)

In [ ]:
!python scripts/05_train_wgan.py --config configs/base.yaml --wgan configs/wgan.yaml \
    --classes-filter "Spondylolysthesis" --out-tag 05_roi_wgan

### 5.5 WGAN — Surgical implant (case_3)

In [ ]:
!python scripts/05_train_wgan.py --config configs/base.yaml --wgan configs/wgan.yaml \
    --classes-filter "Surgical implant" --out-tag 05_roi_wgan

### 5.6 WGAN — Other lesions (case_4)

In [ ]:
!python scripts/05_train_wgan.py --config configs/base.yaml --wgan configs/wgan.yaml \
    --classes-filter "Other lesions" --out-tag 05_roi_wgan

### 5.QA — her sınıfın son sample grid'i

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt, cv2
for cdir in sorted(Path('outputs/05_roi_wgan').glob('case_*')):
    for sdir in sorted(cdir.glob('*')):
        ss=sorted((sdir/'samples').glob('iter_*.png'))
        if not ss: continue
        print(f'{cdir.name}/{sdir.name}: {ss[-1].name}')
        plt.figure(figsize=(5,5)); plt.imshow(cv2.imread(str(ss[-1]),0),cmap='gray'); plt.axis('off'); plt.show()

## 6. Sentetik üretim (Phase 07)

Her sınıf için seçilen checkpoint'ten N=800 yama üret. `CKPT_ITER`'i sample QA'ya göre düzenle (paper: çöküşten önceki snapshot). Varsayılan: 6000.

In [ ]:
import subprocess
CKPT_ITER = {  # sample QA'dan sonra sınıf başına en iyi snapshot'ı buraya yaz
  'disc_space_narrowing':'case_1', 'vertebral_collapse':'case_1',
  'foraminal_stenosis':'case_2', 'spondylolysthesis':'case_2',
  'surgical_implant':'case_3', 'other_lesions':'case_4'}
ITER = 6000   # tüm sınıflar için ortak snapshot; istersen sınıf başına ayır
from pathlib import Path
for slug,case in CKPT_ITER.items():
    ck=f'outputs/05_roi_wgan/{case}/{slug}/checkpoints/iter_{ITER:06d}.pt'
    if not Path(ck).exists():
        cks=sorted(Path(f'outputs/05_roi_wgan/{case}/{slug}/checkpoints').glob('*.pt'))
        ck=str(cks[-1]) if cks else None
    if ck is None: print('NO CKPT', slug); continue
    print('generating', slug, 'from', ck)
    subprocess.run(['python','scripts/07_generate_wgan.py','--checkpoint',ck,'--n-samples','800','--batch-size','64'],check=True)

## 7. ROI Hybrid (real yamalar + WGAN yamalar + 3 transform)

Her case için hybrid set kur (traditional ROI + WGAN üretimleri + WGAN'a transform), sonra eğit.

### 7.1 case_1 — build + train (WGAN: disc_space_narrowing, vertebral_collapse)

In [ ]:
!python scripts/08_build_hybrid_set.py \
    --config configs/base.yaml --cases configs/cases.yaml --case case_1 \
    --wgan-classes disc_space_narrowing vertebral_collapse --variant-tag roi_full \
    --traditional-root outputs/04_roi_traditional
!cat outputs/08_hybrid_roi_full/case_1/summary.md
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --train-csv-root outputs/08_hybrid_roi_full --train-csv-name train_hybrid.csv \
    --splits-tag 02b_roi --cases-filter case_1 \
    --out-tag 08_roi_hybrid

### 7.2 case_2 — build + train (WGAN: foraminal_stenosis, spondylolysthesis)

In [ ]:
!python scripts/08_build_hybrid_set.py \
    --config configs/base.yaml --cases configs/cases.yaml --case case_2 \
    --wgan-classes foraminal_stenosis spondylolysthesis --variant-tag roi_full \
    --traditional-root outputs/04_roi_traditional
!cat outputs/08_hybrid_roi_full/case_2/summary.md
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --train-csv-root outputs/08_hybrid_roi_full --train-csv-name train_hybrid.csv \
    --splits-tag 02b_roi --cases-filter case_2 \
    --out-tag 08_roi_hybrid

### 7.3 case_3 — build + train (WGAN: surgical_implant)

In [ ]:
!python scripts/08_build_hybrid_set.py \
    --config configs/base.yaml --cases configs/cases.yaml --case case_3 \
    --wgan-classes surgical_implant --variant-tag roi_full \
    --traditional-root outputs/04_roi_traditional
!cat outputs/08_hybrid_roi_full/case_3/summary.md
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --train-csv-root outputs/08_hybrid_roi_full --train-csv-name train_hybrid.csv \
    --splits-tag 02b_roi --cases-filter case_3 \
    --out-tag 08_roi_hybrid

### 7.4 case_4 — build + train (WGAN: other_lesions)

In [ ]:
!python scripts/08_build_hybrid_set.py \
    --config configs/base.yaml --cases configs/cases.yaml --case case_4 \
    --wgan-classes other_lesions --variant-tag roi_full \
    --traditional-root outputs/04_roi_traditional
!cat outputs/08_hybrid_roi_full/case_4/summary.md
!python scripts/03_train_classifier.py \
    --config configs/base.yaml --cases configs/cases.yaml --classifier configs/classifier.yaml \
    --train-csv-root outputs/08_hybrid_roi_full --train-csv-name train_hybrid.csv \
    --splits-tag 02b_roi --cases-filter case_4 \
    --out-tag 08_roi_hybrid

## 8. Final karşılaştırma — baseline / traditional / hybrid (ROI), tüm case×backbone

In [ ]:
import json, pandas as pd
from pathlib import Path
def collect(tag,label):
    rows=[]
    for cd in sorted(Path(f'outputs/{tag}').glob('case_*')):
        for bb in sorted(cd.glob('*')):
            mp=bb/'metrics.json'
            if not mp.exists(): continue
            m=json.loads(mp.read_text())
            rows.append({'cond':label,'case':m['case'],'backbone':m['backbone'],
                         'macro_f1':round(m['best_test_macro_f1'],4)})
    return pd.DataFrame(rows)
frames=[collect(t,l) for t,l in [('03_roi_baseline','baseline'),('04_roi_traditional','traditional'),('08_roi_hybrid','hybrid')]]
frames=[f for f in frames if len(f)]
allc=pd.concat(frames,ignore_index=True)
piv=allc.pivot_table(index=['case','backbone'],columns='cond',values='macro_f1')
cols=[c for c in ['baseline','traditional','hybrid'] if c in piv.columns]
piv=piv[cols]
print(piv)
# also vs old whole-image baseline if present
wi=collect('03_baseline','whole_image_baseline')
if len(wi):
    print('\nWhole-image baseline (eski) macro F1:')
    print(wi.pivot_table(index=['case','backbone'],values='macro_f1'))
piv